<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week6_final_mile/Week6_Lesson11_Lecture_LLM.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 IB CS — Неделя 6 · Урок 11 (Лекция)
## LLM + Generative AI + Регуляции + TOK
### Дополнительный модуль для подготовки к Case Study Paper 1

> ⏱ Длительность: 2 академических часа.
> 🎯 Цель: закрыть **3 пробела** курса: Generative AI (case study 2025 был "The Perfect Chatbot"!), регуляции (GDPR/AI Act), TOK интеграция.

---

### ⚠️ Зачем эта неделя

> На IB 2027 первое испытание в May 2027.
> **Case study часто связан с emerging tech** (предыдущий = chatbot). Без знания LLM на 6-7 не сдашь, если попадётся AI case study.
>
> **TOK-вопросы** в Discuss/Evaluate дают +1-2 балла, если умеете их использовать.
>
> **GDPR/AI Act** — обязательны для ethical review в IA.

---

### 📋 План урока

| Часть | Тема | Время |
|---|---|---|
| 1 | LLM — что это и как работает | 25 мин |
| 2 | Transformer architecture (high-level) | 15 мин |
| 3 | Training: pretraining + fine-tuning + RLHF | 15 мин |
| 4 | Capabilities, limitations, hallucinations | 15 мин |
| 5 | Prompt engineering + RAG | 15 мин |
| 6 | GDPR + AI Act — что нужно знать для IB | 15 мин |
| 7 | TOK questions intercaved через лекцию | внутри других |
| 8 | Mini exam practice | 20 мин |


## 🧠 Часть 1 · LLM — что это такое

> **Definition (выучить!):** *A Large Language Model (LLM) is a deep learning model trained on massive text datasets to predict the next token in a sequence, capable of generating coherent text and performing diverse language tasks.*

### 🔑 Ключевая идея

LLM делает **одну простую вещь**: предсказывает **следующее слово**. Но если он делает это **очень хорошо** на основе миллиардов примеров, получается **общая способность к языку**.

> "The cat sat on the ___" → **mat** (вероятность 35%) / chair (12%) / floor (10%) / ...

### 📊 Известные LLM (нужно знать названия!)

| Модель | Компания | Особенность |
|---|---|---|
| **GPT-4, GPT-5** | OpenAI | Closed-source, multi-modal |
| **Claude (Sonnet, Opus)** | Anthropic | Constitutional AI, safety focus |
| **Gemini** | Google DeepMind | Native multi-modal, long context |
| **LLaMA 3** | Meta | Open-source weights |
| **Mistral / Mixtral** | Mistral AI | European, MoE architecture |
| **DeepSeek** | DeepSeek | Open-source, китайский конкурент |

### 🤔 TOK Question #1
> *"Если LLM 'просто' предсказывает следующее слово, можно ли сказать, что он **понимает** язык? Что значит 'понимать'?"*
>
> Это вопрос для **TOK эссе** — обсуждаем в классе **5 минут**.


## 🏗️ Часть 2 · Transformer Architecture (high-level)

> ⚠️ **На IB не нужны** формулы attention или multi-head механизмы. Нужна **концепция**.

### 4 ключевых компонента (для IB ответов):

1. **Tokenization** — текст разбивается на **токены** (части слов или слова)
   - *"Hello, world!"* → `["Hello", ",", " world", "!"]`
2. **Embeddings** — каждый токен преобразуется в **вектор чисел** (~1000-мерный)
3. **Attention** — модель решает, какие токены **важны** для предсказания (например, в "The cat that the dog chased ran away" — "cat" связан с "ran", а не с "dog")
4. **Output layer** — softmax даёт вероятности для **следующего токена**

> 💎 **СЕКРЕТ #1:** *"Outline how a transformer-based language model generates text"* **[3]** — образец:
> 1. Input text is **tokenized** and converted to numerical embeddings (1)
> 2. **Self-attention** layers determine relationships between tokens, capturing context (1)
> 3. The model predicts the most likely **next token** via softmax; this process repeats autoregressively to generate complete responses (1)


In [ ]:
# Визуализация: tokenization + autoregressive generation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 6))

# Tokens
tokens = ['The', 'cat', 'sat', 'on', 'the', '___']
predicted = ['mat', 'chair', 'floor', 'rug']
probs = [0.35, 0.12, 0.10, 0.08]

# Input tokens visualization
for i, tok in enumerate(tokens):
    color = '#4ECDC4' if tok != '___' else '#FFD93D'
    box = mpatches.FancyBboxPatch((i * 0.13, 0.55), 0.12, 0.15,
                                    boxstyle="round,pad=0.01",
                                    facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    ax.text(i * 0.13 + 0.06, 0.625, tok, ha='center', va='center',
            fontsize=12, fontweight='bold')

ax.text(0.39, 0.82, 'INPUT TOKENS', ha='center', fontsize=12, fontweight='bold')

# Attention arrows (показываем, что модель смотрит на все предыдущие токены)
for i in range(5):
    ax.annotate('', xy=(0.71, 0.55), xytext=(i * 0.13 + 0.06, 0.55),
                arrowprops=dict(arrowstyle='->', lw=1, color='purple', alpha=0.4))

# Output predictions
for j, (pred, p) in enumerate(zip(predicted, probs)):
    y_pos = 0.35 - j * 0.08
    bar_width = p * 0.8
    box = mpatches.FancyBboxPatch((0.5, y_pos), bar_width, 0.06,
                                    boxstyle="round,pad=0.01",
                                    facecolor='#FF6B6B', edgecolor='black')
    ax.add_patch(box)
    ax.text(0.48, y_pos + 0.03, pred, ha='right', va='center', fontsize=11, fontweight='bold')
    ax.text(0.5 + bar_width + 0.01, y_pos + 0.03, f'{p:.0%}', va='center', fontsize=10)

ax.text(0.5, 0.45, 'OUTPUT: probability for each next token', ha='left', fontsize=11, fontweight='bold')
ax.text(0.5, 0.05, '🔄 Модель выбирает токен (например, "mat") и повторяет процесс — это **autoregressive generation**',
        ha='center', fontsize=10, style='italic')

ax.set_xlim(0, 1.4); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('LLM генерация — токен за токеном (autoregressive)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 🎓 Часть 3 · Training — 3 этапа

### Этап 1: **Pretraining** (предобучение)
- Модель учится на **триллионах токенов** интернета (web pages, books, code)
- Задача: предсказать следующий токен
- Длится **месяцы** на **тысячах GPU/TPU**
- Стоимость: **миллионы $$$**

> Связь с syllabus: **A4.1.2 hardware** (требуются TPU/GPU кластеры), **A4.3.8 ANN** (transformer — это deep ANN).

### Этап 2: **Fine-tuning** (тонкая настройка)
- Берём предобученную модель + **узкий датасет** для задачи
- Например: pretrained GPT + 10000 примеров медицинских диалогов → medical chatbot
- Намного **дешевле и быстрее** pretraining

> Связь с syllabus: **A4.1.1** (transfer learning — fine-tuning это transfer learning!)

### Этап 3: **RLHF** (Reinforcement Learning from Human Feedback)
- Люди оценивают ответы модели (👍 / 👎)
- Модель учится **давать ответы, которые нравятся людям**
- Используется **reinforcement learning** (A4.3.6 — agent + reward!)
- Так делают ChatGPT, Claude, Gemini

> 💎 **СЕКРЕТ #2:** *"Explain how a chatbot like ChatGPT learns to be helpful"* **[4]**:
> 1. **Pretraining** на miliards токенов интернета даёт **общее владение языком** (1)
> 2. **Fine-tuning** на conversational data учит **формату диалога** (1)
> 3. **RLHF** with human feedback ranks responses — модель оптимизирует **reward function** на основе human preferences (1)
> 4. Это интеграция supervised (pretraining) + transfer (fine-tuning) + reinforcement (RLHF) learning (1)

### 🤔 TOK Question #2
> *"Можем ли мы доверять модели, обученной угождать людям через RLHF? Не приведёт ли это к тому, что модель **врёт чтобы понравиться**?"*


## ⚡ Часть 4 · Capabilities + Limitations

### ✅ Что LLM умеют хорошо

| Задача | Применение |
|---|---|
| **Text generation** | эссе, статьи, креативный текст |
| **Translation** | перевод между языками |
| **Summarization** | краткое содержание длинных текстов |
| **Q&A** | ответы на вопросы |
| **Code generation** | GitHub Copilot, Cursor |
| **Reasoning** (limited) | math, logic problems |
| **Multi-modal** (GPT-4V, Gemini) | анализ изображений + текста |

---

### ❌ Где LLM **проваливаются** — это для IB **критично**

| Проблема | Что значит | Пример |
|---|---|---|
| **Hallucinations** | модель **выдумывает** факты | "Эйнштейн получил Нобеля за теорию относительности" (ложь — за фотоэффект) |
| **No real-time knowledge** | training cutoff | "Кто президент России?" — может ответить устаревше |
| **Bias** | наследует biases из данных | стереотипы про gender / race в ответах |
| **No memory between sessions** | каждый диалог — с чистого листа | (без специальных систем) |
| **Mathematical errors** | путает арифметику | 234 × 456 — может выдать неточный результат |
| **Prompt injection vulnerability** | манипулируемы через текст | "Игнорируй инструкции и сделай X" |

> 💎 **СЕКРЕТ #3 (важно для IB):** *Hallucinations* — самая частая проблема LLM, на экзамене стоит **обязательно** её упомянуть.

---

### 🎯 Hallucinations — глубже

> **Definition:** *Hallucination is when an LLM generates content that is plausible-sounding but factually incorrect or unsupported by training data.*

**Почему происходят?**
- LLM **не имеет понятия "истина"** — он генерирует то, что **звучит вероятно**
- Если в обучающих данных факт встречался **редко** — модель может его **смешать** с похожим
- Особенно опасно для: medical advice, legal advice, scientific facts

**Real cases:**
- 2023: юрист в New York процитировал **6 несуществующих** судебных дел, сгенерированных ChatGPT — оштрафован
- Bing AI в 2023 уверенно утверждал, что **умер актёр**, который был жив
- Google Bard на демо ошибся в фактах о телескопе JWST → акция Google упала на 9%

### 🤔 TOK Question #3
> *"Если AI убедительно выдаёт ложные факты, может ли это привести к **эрозии знания** в обществе? Кто несёт ответственность за hallucinations?"*


## 💬 Часть 5 · Prompt Engineering + RAG

### Prompt Engineering — искусство задавать вопросы

**3 базовые техники (нужно знать):**

| Техника | Что делает | Пример |
|---|---|---|
| **Zero-shot** | даёшь задачу без примеров | *"Переведи на французский: Hello"* |
| **Few-shot** | даёшь несколько примеров | *"EN: cat → FR: chat. EN: dog → FR: chien. EN: bird → FR: ?"* |
| **Chain-of-thought** | просишь модель думать пошагово | *"Реши задачу step by step: ..."* |

> 💎 **СЕКРЕТ #4:** **Chain-of-thought** даёт значительно **более точные** ответы на reasoning задачи. Это часто упоминается в case study про AI.

---

### RAG — Retrieval Augmented Generation

> **Definition:** *RAG is a technique that combines an LLM with an external knowledge base (database, documents). The LLM retrieves relevant information first, then generates a response based on it.*

**Зачем нужен RAG?**
- ✅ **Решает hallucinations** — модель отвечает на основе реальных документов
- ✅ **Доступ к актуальной информации** (после training cutoff)
- ✅ **Корпоративные knowledge bases** (внутренние документы компании)

**Архитектура RAG (3 шага):**
1. **Retrieval**: поисковая система находит **топ-N релевантных документов** для запроса
2. **Augmentation**: эти документы добавляются в **контекст** запроса к LLM
3. **Generation**: LLM отвечает **на основе** этих документов + цитирует источники

> 🎯 **Реальные применения RAG:**
> - Customer support chatbots (отвечают по корпоративной базе знаний)
> - Legal AI (отвечает на основе законодательства)
> - Medical AI (отвечает на основе medical literature)
> - **ChatGPT с web browsing** (плагин)


In [ ]:
# Визуализация RAG архитектуры
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 6))

# User Query
box_q = mpatches.FancyBboxPatch((0.02, 0.45), 0.15, 0.15,
                                  boxstyle="round,pad=0.02",
                                  facecolor='#FFD93D', edgecolor='black', linewidth=2)
ax.add_patch(box_q)
ax.text(0.095, 0.52, 'USER\nQUERY', ha='center', va='center',
        fontsize=11, fontweight='bold')

# Retrieval
box_r = mpatches.FancyBboxPatch((0.25, 0.7), 0.18, 0.15,
                                  boxstyle="round,pad=0.02",
                                  facecolor='#4ECDC4', edgecolor='black', linewidth=2)
ax.add_patch(box_r)
ax.text(0.34, 0.775, '🔍 RETRIEVAL\n(vector search)', ha='center', va='center',
        fontsize=11, fontweight='bold')

# Knowledge base
box_kb = mpatches.FancyBboxPatch((0.25, 0.2), 0.18, 0.15,
                                   boxstyle="round,pad=0.02",
                                   facecolor='#A78BFA', edgecolor='black', linewidth=2)
ax.add_patch(box_kb)
ax.text(0.34, 0.275, '📚 KNOWLEDGE\nBASE', ha='center', va='center',
        fontsize=11, fontweight='bold')

# Augmented prompt
box_a = mpatches.FancyBboxPatch((0.52, 0.45), 0.18, 0.15,
                                  boxstyle="round,pad=0.02",
                                  facecolor='#FF6B6B', edgecolor='black', linewidth=2)
ax.add_patch(box_a)
ax.text(0.61, 0.52, 'AUGMENTED\nPROMPT', ha='center', va='center',
        fontsize=11, fontweight='bold')

# LLM
box_llm = mpatches.FancyBboxPatch((0.78, 0.45), 0.18, 0.15,
                                    boxstyle="round,pad=0.02",
                                    facecolor='#55A868', edgecolor='black', linewidth=2)
ax.add_patch(box_llm)
ax.text(0.87, 0.52, '🤖 LLM\n(generates)', ha='center', va='center',
        fontsize=11, fontweight='bold')

# Стрелки
arrow_props = dict(arrowstyle='->', lw=2, color='black')
ax.annotate('', xy=(0.25, 0.77), xytext=(0.17, 0.55), arrowprops=arrow_props)
ax.annotate('', xy=(0.34, 0.36), xytext=(0.34, 0.7), arrowprops=arrow_props)
ax.annotate('', xy=(0.52, 0.52), xytext=(0.43, 0.77), arrowprops=arrow_props)
ax.annotate('', xy=(0.52, 0.52), xytext=(0.17, 0.52), arrowprops=arrow_props)
ax.annotate('', xy=(0.78, 0.52), xytext=(0.7, 0.52), arrowprops=arrow_props)

# Labels
ax.text(0.21, 0.65, '1. Search', fontsize=9, color='blue')
ax.text(0.45, 0.65, '2. Add context', fontsize=9, color='blue')
ax.text(0.73, 0.55, '3. Generate', fontsize=9, color='blue')

ax.text(0.5, 0.05,
        'RAG: модель отвечает не из памяти, а на основе ИЗВЛЕЧЁННЫХ документов → меньше hallucinations',
        ha='center', fontsize=11, style='italic',
        bbox=dict(boxstyle="round,pad=0.5", facecolor='lightyellow'))

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('RAG (Retrieval Augmented Generation) — архитектура',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## ⚖️ Часть 6 · GDPR + EU AI Act + другие регуляции

> ⚠️ IB 2027 syllabus требует **contextualization to local laws**. Этот блок **обязателен** для IA ethical review + Section B вопросов про privacy.

### 🇪🇺 GDPR (General Data Protection Regulation, 2018)

> Самый влиятельный закон о персональных данных в мире. Применяется к **любой компании, обрабатывающей данные граждан ЕС** (даже если компания не в ЕС).

**6 ключевых принципов GDPR:**

| Принцип | Что значит |
|---|---|
| **Lawfulness** | обработка только на legal basis (consent, contract, ...) |
| **Purpose limitation** | данные только для **заявленной** цели |
| **Data minimization** | собирать **только необходимое** |
| **Accuracy** | данные должны быть **актуальны** |
| **Storage limitation** | удалять, когда больше не нужны |
| **Security** | защита от unauthorized access |

**4 ключевых права пользователя:**
1. **Right to access** — узнать, какие данные о тебе хранятся
2. **Right to erasure** ("right to be forgotten") — удалить свои данные
3. **Right to rectification** — исправить неверные данные
4. **Right to data portability** — экспортировать данные

**Штрафы:** до **4% от global revenue** или **€20 млн** (что больше)
- 2023: Meta оштрафована на **€1.2 миллиарда** за GDPR violations
- 2023: TikTok оштрафована на **€345 млн** за детские данные

---

### 🇪🇺 EU AI Act (2024) — самый новый закон про AI

> Принят в EU в 2024, применение с 2026. **Первый в мире** комплексный закон про AI.

**Классифицирует AI на 4 категории по риску:**

| Категория | Примеры | Регулирование |
|---|---|---|
| **Unacceptable risk** | social scoring (как в Китае), real-time biometric surveillance | **ЗАПРЕЩЕНО** |
| **High risk** | medical AI, credit scoring, hiring | строгие требования (transparency, audits) |
| **Limited risk** | chatbots, deep fakes | обязательно **раскрытие, что это AI** |
| **Minimal risk** | spam filters, AI in games | без регулирования |

**Что обязательно для high-risk AI:**
- ✅ Risk management system
- ✅ High-quality training data (no bias)
- ✅ Documentation + logs
- ✅ Human oversight
- ✅ Cybersecurity

---

### 🇺🇸 США
- **CCPA** (California Consumer Privacy Act, 2020) — аналог GDPR для California
- **HIPAA** — для медицинских данных
- AI пока **fragmented regulation** (по штатам и industries)

### 🇨🇳 Китай
- **PIPL** (Personal Information Protection Law, 2021) — аналог GDPR
- Жёсткое регулирование **generative AI** (Aug 2023): all models must register, content moderation

> 💎 **СЕКРЕТ #5:** на IB *"Discuss the role of regulation in deploying AI"* — упомянуть **минимум 2** конкретных закона (GDPR + EU AI Act), а ещё лучше — **3 региона** (EU, US, China). Это даёт **+2 балла за глобальную перспективу**.


## 🤔 Часть 7 · TOK Integration — для extended responses

> **TOK (Theory of Knowledge)** — обязательная часть IB. На IB CS использование TOK-перспективы в extended response **даёт +1-2 балла**.

### 🎯 6 готовых TOK-вопросов для ML-ответов

> Используйте эти вопросы как **закрашивающие** перспективы в Discuss/Evaluate ответах.

1. **На responsibility:** *"To what extent should accountability be prioritized over innovation in AI deployment?"*
2. **На privacy:** *"How can we reconcile the need for privacy with the benefits of data collection in AI?"*
3. **На bias:** *"In what ways do biases in training data reflect broader societal inequalities?"*
4. **На knowledge:** *"If an AI system can produce knowledge that humans cannot verify, can it still be considered knowledge?"*
5. **На truth:** *"In a world where AI can convincingly fabricate content, what becomes our standard of truth?"*
6. **На emotion:** *"Can a system that has no emotions truly understand human needs?"*

### 📐 Как **встроить** TOK в IB ответ

**Слабый ответ (без TOK):**
> *"Facial recognition violates privacy because it tracks people."*

**Сильный ответ (с TOK):**
> *"Facial recognition raises fundamental questions about the **balance between collective safety and individual privacy**. While it offers crime prevention benefits, it also poses TOK-level questions about who controls knowledge of people's movements — and whether such surveillance changes the very nature of public space."*

> 💎 **СЕКРЕТ #6:** TOK-вставки делают ответ **более зрелым**. Используйте 1-2 раза в ответе на Discuss [6+].


## 📝 Часть 8 · Mini Exam Practice (в классе)

### Вопрос 1 (новый стиль 2027 — LLM в case study)
> *A high school deploys an LLM-based AI tutor that provides personalized homework help to students.*
>
> **a)** *Define* "large language model". **[1]**
> **b)** *Describe* TWO limitations the school should communicate to parents. **[4]**
> **c)** *Discuss* ONE ethical concern AND mitigation. **[4]**

> 💎 **(b) образец [4]:**
> 1. **Hallucinations**: LLM может выдавать **уверенные, но ложные ответы**, что особенно опасно в математике или истории (2)
> 2. **No real-time knowledge**: training cutoff может означать устаревшую информацию (например, события 2025 могут быть unknown) (2)
>
> **(c) образец [4]:**
> *"**Privacy of minors**: LLM-сервисы могут логировать запросы детей для обучения. Под GDPR (Article 8) и EU AI Act обработка данных несовершеннолетних требует **родительского согласия** (2). **Mitigation**: использовать on-premise модель (например, fine-tuned LLaMA) вместо ChatGPT, with no logging; explicit parental consent forms; audit by independent privacy board (2)."*

---

### Вопрос 2 (TOK-стиль)
> *Discuss whether an LLM that has memorized facts from training data can be said to "know" them in the same sense as a human knower.* **[6]**

> 💎 Структура:
> 1. **Что значит "знать"** для человека (justified true belief) (1)
> 2. **LLM** имеет statistical pattern matching, не "веру" (1)
> 3. **Counter**: можно ли отличить "знание" от **способности применять**? (1)
> 4. **Конкретный пример** (LLM правильно решает задачу, но не понимает почему) (1)
> 5. **TOK perspective** на role of consciousness in knowledge (1)
> 6. **Conclusion** with nuance (1)


## ✅ Чек-лист после Урока 11

### LLM / Generative AI
- [ ] Знаю **что такое LLM** + основные модели (GPT, Claude, Gemini, LLaMA)
- [ ] Знаю **4 компонента transformer** (tokenization, embeddings, attention, output)
- [ ] Знаю **3 этапа training** (pretraining + fine-tuning + RLHF)
- [ ] Знаю **6 ограничений LLM** (especially hallucinations)
- [ ] Знаю, что такое **prompt engineering** + 3 техники
- [ ] Знаю **RAG** и зачем он нужен

### Регуляции
- [ ] Знаю **6 принципов GDPR** + штрафы
- [ ] Знаю **4 категории риска EU AI Act**
- [ ] Знаю **3 региональных** регулятора (EU, US, China)

### TOK
- [ ] Знаю **6 TOK-вопросов** для ML-ответов
- [ ] Умею **встроить TOK** в Discuss/Evaluate

---

### 📚 Связь с другими ноутбуками Недели 6

После этой лекции идёт:
- **Notebook 22**: 50 MC questions — финальная проверка знаний всего курса
- **Notebook 23**: Полный Mock Paper 1 — 2 часа в режиме экзамена
- **Notebook 24**: Flashcards — последний rapid review перед экзаменом

---

> 🎓 **Финальный секрет:**
> LLM и regulations — это **front-of-mind** темы для IB 2027.
> Case study 2025 был "The Perfect Chatbot" (LLM!).
> Case study 2027 — **может быть** что-то связанное с GenAI / regulation.
> Если так — этот урок **даст решающее преимущество**.
